In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from essentials import resample_data, three_class_labels, two_class_labels, normalize
from acc_features import GenerateFeatures
import copy

In [2]:
with open('data/df_dict_imu.pkl', 'rb') as f:
    imu_dict = pickle.load(f)
with open('data/df_minze_dict.pkl', 'rb') as f:
    ground_truth_dict = pickle.load(f)
with open('data/df_dict_urineestimate_method1.pkl', 'rb') as f:
    urine_estimate_dict = pickle.load(f)

Remove the data with the almost the entire data being void.
1. subj_9_void4
2. subj_11_void2

In [3]:
del imu_dict['subj_9_void4']
del imu_dict['subj_11_void2']

In [4]:
resampled_imu_dict = resample_data(imu_dict, 70)

41it [00:00, 529.89it/s]


Extract windowed features from the 3 axis of the accelerometer and the acceleration magnitude.
1. mean
2. rms
3. std
4. range
5. variance
6. min
7. max
8. time energy
9. spectral energy
10. permutation entropy
11. spectral entropy

## Data 1
classes: pre-void, void, post-void

fs: 70

sliding window: 1s

overlap: 0.8

filename: three_class_up_1s.csv

In [5]:
labelled_imu_dict = {}
dict = copy.deepcopy(resampled_imu_dict)
for i_void_instance, void_instance in tqdm(enumerate(dict.keys()), desc="Adding labels to IMU data"):
    acc = dict[void_instance]
    gt = ground_truth_dict[void_instance]
    
    # normalize data
    df = normalize(acc)
    
    # add the labels
    labelled_df = three_class_labels(df, gt)

    labelled_imu_dict[void_instance] = labelled_df

Adding labels to IMU data: 41it [00:00, 248.17it/s]


In [6]:
#  do not upsample
all_features = []
for exp_id, imu_data in enumerate(labelled_imu_dict.keys()):    
    # Extract features
    
    df = labelled_imu_dict[imu_data]
    fs = 1 / df['time'].diff().median()
    analyzer = GenerateFeatures(70, window_duration=1.0, overlap=0.8)
    features, _ = analyzer.analyze_multi_axis_imu(labelled_imu_dict[imu_data])

    table = analyzer.create_summary_table()
    table['experiment_id'] = exp_id + 1  # Track source
    all_features.append(table)
    

final_features = pd.concat(all_features, ignore_index=True)

Analyzing : 100%|██████████| 4/4 [00:01<00:00,  2.24it/s]


In [8]:
final_features.to_csv('feature_based_data/three_class_up_1s.csv', index=False)
final_features.head()

,center_time,start_time,end_time,acc_x_permutation_entropy,acc_x_spectral_entropy,acc_x_mean,acc_x_std,acc_x_range,acc_x_rms,acc_x_var,...,acc_mag_std,acc_mag_range,acc_mag_rms,acc_mag_var,acc_mag_min,acc_mag_max,acc_mag_time_energy,acc_mag_spectral_energy,label,experiment_id
0,0.500000,0.000000,1.000000,0.865372,0.499289,0.936305,1.137581,5.41962,1.473349,1.294090,...,0.952641,4.247046,1.995090,0.907525,0.168266,4.415312,278.626807,19503.876521,pre-void,1
1,0.685714,0.185714,1.185714,0.816043,0.481647,0.989842,1.158399,5.41962,1.523704,1.341888,...,0.972284,4.247046,1.987835,0.945337,0.168266,4.415312,276.604259,19362.298164,pre-void,1
2,0.871429,0.371429,1.371429,0.822487,0.536058,1.195631,1.187488,5.41962,1.685130,1.410128,...,1.028004,4.247046,2.171321,1.056792,0.168266,4.415312,330.024516,23101.716142,pre-void,1
3,1.057143,0.557143,1.557143,0.792757,0.535459,1.251095,1.199598,5.41962,1.733284,1.439036,...,1.007495,4.247046,2.172495,1.015047,0.168266,4.415312,330.381288,23126.690133,pre-void,1
4,1.242857,0.742857,1.742857,0.792624,0.544907,1.165332,1.209628,5.41962,1.679642,1.463199,...,1.079930,4.247046,2.106117,1.166249,0.168266,4.415312,310.501137,21735.079605,pre-void,1


## Data 2
classes: pre-void, void, post-void

fs: 70

sliding window: 2s

overlap: 0.8

filename: three_class_up_2s.csv

In [9]:
#  do not upsample
all_features = []
for exp_id, imu_data in enumerate(labelled_imu_dict.keys()):    
    # Extract features
    
    df = labelled_imu_dict[imu_data]
    fs = 1 / df['time'].diff().median()
    analyzer = GenerateFeatures(70, window_duration=2.0, overlap=0.8)
    features, _ = analyzer.analyze_multi_axis_imu(labelled_imu_dict[imu_data])

    table = analyzer.create_summary_table()
    table['experiment_id'] = exp_id + 1  # Track source
    all_features.append(table)
    

final_features = pd.concat(all_features, ignore_index=True)

Analyzing : 100%|██████████| 4/4 [00:00<00:00,  5.45it/s]


In [10]:
final_features.to_csv('feature_based_data/three_class_up_2s.csv', index=False)
final_features.head()

,center_time,start_time,end_time,acc_x_permutation_entropy,acc_x_spectral_entropy,acc_x_mean,acc_x_std,acc_x_range,acc_x_rms,acc_x_var,...,acc_mag_std,acc_mag_range,acc_mag_rms,acc_mag_var,acc_mag_min,acc_mag_max,acc_mag_time_energy,acc_mag_spectral_energy,label,experiment_id
0,1.000000,0.000000,2.000000,0.838257,0.640434,1.027166,1.006075,5.419620,1.437796,1.012187,...,0.910694,4.247046,1.881658,0.829364,0.168266,4.415312,495.688987,69396.458205,pre-void,1
1,1.385714,0.385714,2.385714,0.837870,0.624751,1.050145,0.936625,5.419620,1.407149,0.877266,...,0.881961,4.247046,1.783063,0.777855,0.168266,4.415312,445.103663,62314.512886,pre-void,1
2,1.771429,0.771429,2.771429,0.830795,0.635881,0.996281,0.888150,5.419620,1.334686,0.788810,...,0.847308,4.247046,1.714364,0.717932,0.168266,4.415312,411.466194,57605.267222,pre-void,1
3,2.157143,1.157143,3.157143,0.836304,0.629307,0.931380,0.584322,3.559709,1.099500,0.341432,...,0.587919,3.197414,1.424697,0.345648,0.407560,3.604973,284.166609,39783.325254,pre-void,1
4,2.542857,1.542857,3.542857,0.859942,0.753119,0.764054,0.266251,1.494012,0.809116,0.070890,...,0.268105,1.299565,1.116902,0.071880,0.582435,1.882000,174.645666,24450.393183,pre-void,1


## Add Gyroscope

## Data 3
classes: pre-void, void, post-void

fs: 70

sliding window: 1s

overlap: 0.8

filename: three_class_up_1s_ag.csv

In [12]:
from acc_gyr_features import GenerateFeatures

In [13]:
#  do not upsample
all_features = []
for exp_id, imu_data in enumerate(labelled_imu_dict.keys()):    
    # Extract features
    
    df = labelled_imu_dict[imu_data]
    fs = 1 / df['time'].diff().median()
    analyzer = GenerateFeatures(70, window_duration=1.0, overlap=0.8)
    features, _ = analyzer.analyze_multi_axis_imu(labelled_imu_dict[imu_data])

    table = analyzer.create_summary_table()
    table['experiment_id'] = exp_id + 1  # Track source
    all_features.append(table)
    

final_features = pd.concat(all_features, ignore_index=True)

Analyzing : 100%|██████████| 8/8 [00:03<00:00,  2.49it/s]


In [14]:
final_features.to_csv('feature_based_data/three_class_up_1s_ag.csv', index=False)
final_features.head()

,center_time,start_time,end_time,acc_x_permutation_entropy,acc_x_spectral_entropy,acc_x_mean,acc_x_std,acc_x_range,acc_x_rms,acc_x_var,...,gyr_mag_std,gyr_mag_range,gyr_mag_rms,gyr_mag_var,gyr_mag_min,gyr_mag_max,gyr_mag_time_energy,gyr_mag_spectral_energy,label,experiment_id
0,0.500000,0.000000,1.000000,0.865372,0.499289,0.936305,1.137581,5.41962,1.473349,1.294090,...,1.539962,6.482408,3.189520,2.371482,0.582247,7.064655,712.112445,49847.871131,pre-void,1
1,0.685714,0.185714,1.185714,0.816043,0.481647,0.989842,1.158399,5.41962,1.523704,1.341888,...,1.625116,6.482408,3.364800,2.641003,0.582247,7.064655,792.531701,55477.219040,pre-void,1
2,0.871429,0.371429,1.371429,0.822487,0.536058,1.195631,1.187488,5.41962,1.685130,1.410128,...,1.765261,6.822305,3.627525,3.116145,0.895522,7.717826,921.125513,64478.785876,pre-void,1
3,1.057143,0.557143,1.557143,0.792757,0.535459,1.251095,1.199598,5.41962,1.733284,1.439036,...,1.637352,6.822305,3.470356,2.680920,0.895522,7.717826,843.036106,59012.527412,pre-void,1
4,1.242857,0.742857,1.742857,0.792624,0.544907,1.165332,1.209628,5.41962,1.679642,1.463199,...,1.719650,7.160302,3.425538,2.957196,0.557524,7.717826,821.401786,57498.125018,pre-void,1


## Data 4
classes: pre-void, void, post-void

fs: 70

sliding window: 2s

overlap: 0.8

filename: three_class_up_2s_ag.csv

In [15]:
#  do not upsample
all_features = []
for exp_id, imu_data in enumerate(labelled_imu_dict.keys()):    
    # Extract features
    
    df = labelled_imu_dict[imu_data]
    fs = 1 / df['time'].diff().median()
    analyzer = GenerateFeatures(70, window_duration=2.0, overlap=0.8)
    features, _ = analyzer.analyze_multi_axis_imu(labelled_imu_dict[imu_data])

    table = analyzer.create_summary_table()
    table['experiment_id'] = exp_id + 1  # Track source
    all_features.append(table)
    

final_features = pd.concat(all_features, ignore_index=True)

Analyzing : 100%|██████████| 8/8 [00:01<00:00,  5.84it/s]


In [16]:
final_features.to_csv('feature_based_data/three_class_up_2s_ag.csv', index=False)
final_features.head()

,center_time,start_time,end_time,acc_x_permutation_entropy,acc_x_spectral_entropy,acc_x_mean,acc_x_std,acc_x_range,acc_x_rms,acc_x_var,...,gyr_mag_std,gyr_mag_range,gyr_mag_rms,gyr_mag_var,gyr_mag_min,gyr_mag_max,gyr_mag_time_energy,gyr_mag_spectral_energy,label,experiment_id
0,1.000000,0.000000,2.000000,0.838257,0.640434,1.027166,1.006075,5.419620,1.437796,1.012187,...,1.743709,7.437734,3.088431,3.040523,0.280092,7.717826,1335.376699,186952.737877,pre-void,1
1,1.385714,0.385714,2.385714,0.837870,0.624751,1.050145,0.936625,5.419620,1.407149,0.877266,...,1.787156,7.554312,2.801069,3.193925,0.163515,7.717826,1098.437930,153781.310154,pre-void,1
2,1.771429,0.771429,2.771429,0.830795,0.635881,0.996281,0.888150,5.419620,1.334686,0.788810,...,1.739820,7.636681,2.451882,3.026974,0.081146,7.717826,841.641483,117829.807593,pre-void,1
3,2.157143,1.157143,3.157143,0.836304,0.629307,0.931380,0.584322,3.559709,1.099500,0.341432,...,1.502188,7.636681,1.943241,2.256569,0.081146,7.717826,528.665989,74013.238410,pre-void,1
4,2.542857,1.542857,3.542857,0.859942,0.753119,0.764054,0.266251,1.494012,0.809116,0.070890,...,0.503504,3.593582,0.790187,0.253516,0.074431,3.668013,87.415349,12238.148920,pre-void,1
